# Automotive RAG Question Answering System - Phase 2

This notebook demonstrates the Phase 2 Retrieval-Augmented Generation QA System.

**Architecture**:
`Question` -> `Retriever` -> `Retrieved Chunks` -> `Prompt Builder` -> `LLM Engine` -> `Grounded Answer` -> `Source Traceability` -> `RAG Evaluator`

In [ ]:
!pip install -r ../requirements.txt

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.processing import process_document
from src.chunking import recursive_chunking
from src.embeddings import generate_embeddings
from src.vector_store import VectorStore
from src.retriever import Retriever
from src.rag_engine import AutomotiveRAG
from src.rag_evaluator import RAGEvaluator

In [ ]:
# Step 1 & 2: Load Document and Build Index
file_path = '../test_assets/sample.pdf'
text, metadata = process_document(file_path)
chunks = recursive_chunking(text, chunk_size=500)
metadatas = [metadata.copy() for _ in chunks]

embeddings = generate_embeddings(chunks)

vs = VectorStore()
vs.build_index(embeddings, chunks, metadatas)
retriever = Retriever(vs)

In [ ]:
# Step 3: Ask Question
rag = AutomotiveRAG(retriever)
result = rag.ask("What is the content of the PDF?")

print("\n=== RAG Answer ===")
print(result['answer'])

print("\n=== Traceability ===")
for s in result['sources']:
    print(s)

print(f"\nRetrieval Latency: {result['retrieval_time_ms']:.2f} ms")

evaluator = RAGEvaluator()
print(f"Answer Found: {evaluator.answer_found(result)}")
print(f"Sources Count: {evaluator.source_count(result)}")